In [1]:
from rdkit import Chem

# Đường dẫn file gốc bị lỗi
input_path = r"d:\dockingdemo1\mynuu_docking\mynuu_docking_pipeline\results\LIG_mynuu_REAL_01_3D_S_full_isomer_preopt.sdf"
# Đường dẫn file mới đã fix
output_path = r"d:\dockingdemo1\mynuu_docking\mynuu_docking_pipeline\results\LIG_mynuu_S_fixed_full_final.sdf"

suppl = Chem.SDMolSupplier(input_path, removeHs=False)
writer = Chem.SDWriter(output_path)

valid_count = 0
for i, mol in enumerate(suppl):
    if mol is None:
        continue
    try:
        # Ép RDKit tính toán lại tính thơm và gán lại liên kết
        Chem.SanitizeMol(mol) 
        writer.write(mol)
        valid_count += 1
    except:
        print(f"⚠️ Conformer {i} vẫn lỗi, bỏ qua.")

writer.close()
print(f"✅ Đã sửa xong! Lưu được {valid_count} conformers vào {output_path}")

✅ Đã sửa xong! Lưu được 10 conformers vào d:\dockingdemo1\mynuu_docking\mynuu_docking_pipeline\results\LIG_mynuu_S_fixed_full_final.sdf


In [ ]:
# 🧬 COVALENT DOCKING PIPELINE v1.0
"""
**Boron Warhead Covalent Docking for LIG_0001 (PHASE 3)**

This Jupyter notebook orchestrates high-throughput covalent molecular docking using GNINA.

**Pipeline Architecture:**
- PHASE 1: Pre-optimize boron geometry (MMFF94s) ✅ Complete
- PHASE 2: Identify covalent bonding sites (nucleophiles) ✅ Complete  
- PHASE 3: Generate & execute covalent docking (THIS NOTEBOOK) 🔄 In Progress
- PHASE 4-6: Analyze results & generate publication report (Future)

**Execution Model:** Sequential (target-by-target, conformer-by-conformer)

**Output:** 640 total docking runs (8 targets / 69 nucleophiles/ 2 warheads /20 conformers)
"""
## Cell 1: Configuration & Setup

#Load .env configuration and initialize global variables.
# ========================================
# IMPORTS
# ========================================
import os
import subprocess
import time
import json
import csv
import shutil
import socket
import traceback
from pathlib import Path
from typing import List, Dict, Tuple, Optional
from datetime import datetime

# Data processing
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem
from Bio.PDB import PDBParser, PDBIO
from Bio.PDB.Polypeptide import is_aa

# Configuration & reporting
from dotenv import load_dotenv
from openpyxl import Workbook, load_workbook
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side

# Display
from IPython.display import display, HTML, Markdown

print("✅ All imports successful")
# ========================================
# CONFIGURATION LOADING
# ========================================

NOTEBOOK_DIR = Path.cwd()
print(f"📂 Notebook directory: {NOTEBOOK_DIR}")

# Load .env file
_env_file = NOTEBOOK_DIR / ".env"
if _env_file.exists():
    load_dotenv(_env_file, override=True)
    print(f"✅ Loaded .env from {_env_file}")
else:
    print(f"⚠️  No .env file found at {_env_file}")

# ========================================
# PATH RESOLUTION FUNCTION
# ========================================
def _resolve_path(env_var: str, fallback: str) -> str:
    """Resolve environment variables to absolute paths with fallbacks."""
    raw = os.environ.get(env_var, fallback)
    expanded = os.path.expanduser(raw)
    if not os.path.isabs(expanded):
        expanded = str(NOTEBOOK_DIR / expanded)
    return os.path.normpath(expanded)

# ========================================
# GLOBAL CONFIGURATION
# ========================================

# Base directories
BASE_DIR = _resolve_path("DOCKING_BASE_DIR", str(NOTEBOOK_DIR))
RESULTS_BASE_DIR = _resolve_path("RESULTS_BASE_DIR", f"{BASE_DIR}/results/phase3_docking_results")

# GNINA executable
GNINA_BIN = _resolve_path("GNINA_BIN", "/usr/local/bin/gnina")
GNINA_TIMEOUT_SEC = int(os.environ.get("GNINA_TIMEOUT_SEC", "3600"))
GPU_DEVICE = os.environ.get("GNINA_GPU_DEVICE", "0")

# Input files
PROTEIN_PATH = _resolve_path("PROTEIN_PATH", f"{BASE_DIR}/data/receptor.pdb")
REF_LIGAND = _resolve_path("REF_LIGAND", f"{BASE_DIR}/data/ref_ligand.sdf")
LIGAND_SDF = _resolve_path("LIGAND_SDF", f"{BASE_DIR}/data/ligand_conformers.sdf")
NUCLEOPHILE_DATA_PATH = _resolve_path("NUCLEOPHILE_DATA_PATH", f"{NOTEBOOK_DIR}/phase2_nucleophile_data.json")

# Docking parameters
FLEX_RESIDUES = os.environ.get("FLEX_RESIDUES", "A:253,A:256,A:257,A:259")
NUM_MODES = int(os.environ.get("NUM_MODES", "10"))
EXHAUSTIVENESS = int(os.environ.get("EXHAUSTIVENESS", "64"))
SEED = os.environ.get("SEED", "42")
AUTOBOX_ADD = float(os.environ.get("AUTOBOX_ADD", "5.0"))  # Cast to float at load
AUTOBOX_EXTEND = int(os.environ.get("AUTOBOX_EXTEND", "1"))  # Cast to int at load

# Covalent docking parameters
WARHEAD_B1_PATTERN = os.environ.get("WARHEAD_B1_SMARTS", "[B;R1;r5](OC)N")
WARHEAD_B2_PATTERN = os.environ.get("WARHEAD_B2_SMARTS", "[B;R1;r6](O)OC")
COVALENT_BOND_DISTANCE_MIN = float(os.environ.get("COVALENT_BOND_DISTANCE_MIN", "1.5"))
COVALENT_BOND_DISTANCE_MAX = float(os.environ.get("COVALENT_BOND_DISTANCE_MAX", "2.4"))
COVALENT_FORMATION_SCORE_MIN = float(os.environ.get("COVALENT_FORMATION_SCORE_MIN", "0.70"))
AFFINITY_POOR_THRESHOLD = float(os.environ.get("AFFINITY_POOR_THRESHOLD", "-6.5"))

# Covalent reactor atom target (from .env) - DO NOT HARDCODE
COVALENT_REC_ATOM = os.environ.get("COVALENT_REC_ATOM", "A:279:OG1")
COVALENT_LIG_ATOM_PATTERN = os.environ.get("COVALENT_LIG_ATOM_PATTERN", "[B;R1;r5](OC)N")
COVALENT_LIG_ATOM_POSITION_STR = os.environ.get("COVALENT_LIG_ATOM_POSITION", "")
COVALENT_LIG_ATOM_POSITION = None
if COVALENT_LIG_ATOM_POSITION_STR.strip():
    try:
        coords = [float(x.strip()) for x in COVALENT_LIG_ATOM_POSITION_STR.split(",")]
        if len(coords) == 3:
            COVALENT_LIG_ATOM_POSITION = tuple(coords)
    except ValueError:
        pass  # Use None if parsing fails
COVALENT_FIX_LIG_ATOM_POSITION = int(os.environ.get("COVALENT_FIX_LIG_ATOM_POSITION", "0")) == 1
COVALENT_BOND_ORDER = int(os.environ.get("COVALENT_BOND_ORDER", "1"))
COVALENT_OPTIMIZE_LIG = False # True thi hien tai dang bi loi phan mem

# Status constants
STATUS_PENDING = "PENDING"
STATUS_RUNNING = "RUNNING"
STATUS_DONE = "DONE"
STATUS_FAILED = "FAILED"
STATUS_DONE_WITH_WARNING = "DONE_WITH_WARNING"

print("\n" + "=" * 80)
print("🧬 COVALENT DOCKING PIPELINE v1.0")
print("=" * 80)
print(f"📂 Base directory:        {BASE_DIR}")
print(f"🔬 GNINA binary:          {GNINA_BIN}")
print(f"🧪 Protein:               {PROTEIN_PATH}")
print(f"📦 Ligand conformers:     {LIGAND_SDF}")
print(f"🔬 Nucleophile data:      {NUCLEOPHILE_DATA_PATH}")
print(f"📁 Output directory:      {RESULTS_BASE_DIR}")
print(f"\n⚙️  Parameters:")
print(f"   - Covalent receptor atom: {COVALENT_REC_ATOM}")
print(f"   - Covalent ligand pattern: {COVALENT_LIG_ATOM_PATTERN}")
print(f"   - Covalent bond distance: {COVALENT_BOND_DISTANCE_MIN}-{COVALENT_BOND_DISTANCE_MAX}Å")
print(f"   - Formation score threshold: >{COVALENT_FORMATION_SCORE_MIN:.2f}")
print(f"   - Optimize ligand: {'Yes' if COVALENT_OPTIMIZE_LIG else 'No'}")
print(f"   - GNINA timeout: {GNINA_TIMEOUT_SEC}s")
print(f"   - GPU device: {GPU_DEVICE}")
print(f"   - Poses per run: {NUM_MODES}")
print(f"   - Exhaustiveness: {EXHAUSTIVENESS}")
print("=" * 80 + "\n")

# ========================================
# VALIDATION: Check environment configuration
# ========================================
print("\n🔍 Validating environment configuration...")

# 1. Check numeric parameters have no units
validation_errors = []

try:
    # These should be pure numbers, no units like \"A\" or \"Å\"
    test_float = float(AUTOBOX_ADD)
    test_float = float(AUTOBOX_EXTEND)
    test_float = float(COVALENT_BOND_DISTANCE_MIN)
    test_float = float(COVALENT_BOND_DISTANCE_MAX)
    test_float = float(COVALENT_FORMATION_SCORE_MIN)
    test_float = float(AFFINITY_POOR_THRESHOLD)
    test_int = int(NUM_MODES)
    test_int = int(EXHAUSTIVENESS)
    test_int = int(GNINA_TIMEOUT_SEC)
    print("✅ All numeric parameters valid (no units)")
except ValueError as e:
    validation_errors.append(f"❌ Numeric parameter parsing failed: {e}\\n   Check .env file - parameters should not have units (e.g., use '5.0' not '5.0A')")

# 2. Check COVALENT_REC_ATOM format
if COVALENT_REC_ATOM:
    parts = COVALENT_REC_ATOM.split(':')
    if len(parts) != 3:
        validation_errors.append(f"❌ COVALENT_REC_ATOM format invalid: '{COVALENT_REC_ATOM}'\\n   Expected format: 'CHAIN:RESNUM:ATOM' (e.g., 'A:248:SG')")
    else:
        try:
            int(parts[1])  # resnum should be integer
            print(f"✅ COVALENT_REC_ATOM format valid: {COVALENT_REC_ATOM}")
        except ValueError:
            validation_errors.append(f"❌ COVALENT_REC_ATOM residue number not integer: '{parts[1]}'")

# 3. Check file paths exist
if PROTEIN_PATH and not os.path.exists(PROTEIN_PATH):
    validation_errors.append(f"❌ PROTEIN_PATH not found: {PROTEIN_PATH}")
else:
    print(f"✅ PROTEIN_PATH exists: {PROTEIN_PATH}")

if LIGAND_SDF and not os.path.exists(LIGAND_SDF):
    validation_errors.append(f"❌ LIGAND_SDF not found: {LIGAND_SDF}")
else:
    print(f"✅ LIGAND_SDF exists: {LIGAND_SDF}")

if NUCLEOPHILE_DATA_PATH and not os.path.exists(NUCLEOPHILE_DATA_PATH):
    validation_errors.append(f"❌ NUCLEOPHILE_DATA_PATH not found: {NUCLEOPHILE_DATA_PATH}")
else:
    print(f"✅ NUCLEOPHILE_DATA_PATH exists: {NUCLEOPHILE_DATA_PATH}")

if RESULTS_BASE_DIR and not os.path.exists(RESULTS_BASE_DIR):
    print(f"⚠️  RESULTS_BASE_DIR will be created: {RESULTS_BASE_DIR}")
else:
    print(f"✅ RESULTS_BASE_DIR exists: {RESULTS_BASE_DIR}")

# 4. Check GNINA binary exists
if GNINA_BIN and not os.path.exists(GNINA_BIN):
    validation_errors.append(f"❌ GNINA_BIN not found: {GNINA_BIN}\\n   Please ensure GNINA is installed and path is correct.")
else:
    print(f"✅ GNINA_BIN exists: {GNINA_BIN}")

# 5. Print summary
print("\n" + "=" * 80)
if validation_errors:
    print("❌ VALIDATION FAILED - Configuration errors found:")
    for error in validation_errors:
        print(f"  {error}")
    print("=" * 80)
    raise RuntimeError("Please fix configuration errors before running pipeline")
else:
    print("✅ All validations passed - Configuration is correct!")
    print("=" * 80)

# ========================================
# PROTEIN PATH RESOLUTION (Multi-Target)
# ========================================

def get_protein_path_for_target(target_name: str) -> str:
    """
    Dynamically resolve protein PDB path for a specific target.
    
    Args:
        target_name: Target identifier (e.g., 'PPARA_7BQ2')
    
    Returns:
        Full path to the prepared protein PDB file
    
    Raises:
        ValueError: If target not found in TARGET_MAPPING
    """
    # First, establish the protein data directory
    PROTEIN_DATA_DIR = _resolve_path("PROTEIN_DATA_DIR", f"{BASE_DIR}/data")
    
    mapping_key = f"TARGET_MAPPING_{target_name}"
    filename = os.environ.get(mapping_key)
    
    if not filename:
        raise ValueError(
            f"Target '{target_name}' not found in environment. "
            f"Please set {mapping_key} in .env file."
        )
    
    full_path = os.path.join(PROTEIN_DATA_DIR, filename)
    
    if not os.path.exists(full_path):
        raise ValueError(
            f"Protein file not found: {full_path}\n"
            f"Check that {mapping_key}={filename} in .env is correct."
        )
    
    return full_path

## Cell 2: Utility Functions

#Core functions for status management, ligand parsing, and score extraction (adapted from flexible docking v2.6.2 + new covalent-specific functions).
# ========================================
# SUBPROCESS ENVIRONMENT SETUP
# ========================================

def _get_subprocess_env() -> dict:
    """Setup subprocess environment with CUDA and library paths."""
    env = os.environ.copy()
    
    # Add Conda libraries to LD_LIBRARY_PATH (Linux/WSL only)
    if os.name != 'nt':  # Not Windows
        conda_prefix = env.get("CONDA_PREFIX", "")
        if conda_prefix:
            conda_lib = os.path.join(conda_prefix, "lib")
            current_ld = env.get("LD_LIBRARY_PATH", "")
            if conda_lib not in current_ld:
                env["LD_LIBRARY_PATH"] = (
                    f"{conda_lib}:{current_ld}" if current_ld else conda_lib
                )
    
    # Pass through CUDA_VISIBLE_DEVICES
    cuda_vis = os.environ.get("CUDA_VISIBLE_DEVICES")
    if cuda_vis is not None:
        env["CUDA_VISIBLE_DEVICES"] = cuda_vis
    
    return env

# ========================================
# STATUS MANAGEMENT (PHASE 1-2 PATTERN)
# ========================================

def write_status(lig_root: str, status: str, **kwargs):
    """Write status file with metadata."""
    status_file = os.path.join(lig_root, "STATUS.txt")
    
    if status == STATUS_RUNNING:
        with open(status_file, "w") as f:
            f.write(f"STATUS={status}\n")
            f.write(f"HOST={socket.gethostname()}\n")
            f.write(f"START_TIME={datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    else:
        with open(status_file, "a") as f:
            f.write(f"STATUS={status}\n")
            for key, value in kwargs.items():
                f.write(f"{key.upper()}={value}\n")
            f.write(f"END_TIME={datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

def read_status(lig_root: str) -> str:
    """Read current status from status file."""
    status_file = os.path.join(lig_root, "STATUS.txt")
    if not os.path.exists(status_file):
        return STATUS_PENDING
    
    last_status = STATUS_PENDING
    try:
        with open(status_file, "r") as f:
            for line in f:
                if line.startswith("STATUS="):
                    last_status = line.strip().split("=", 1)[1]
    except Exception:
        pass
    
    return last_status

def get_status_details(lig_root: str) -> dict:
    """Extract all status metadata."""
    status_file = os.path.join(lig_root, "STATUS.txt")
    details = {}
    
    if os.path.exists(status_file):
        try:
            with open(status_file, "r") as f:
                for line in f:
                    if "=" in line:
                        key, value = line.strip().split("=", 1)
                        details[key.lower()] = value
        except Exception:
            pass
    
    return details

print("✅ Status management functions loaded")
# ========================================
# UTILITY FUNCTIONS
# ========================================

def sanitize_name(name: str, max_len: int = 80) -> str:
    """Sanitize string for use as directory/filename."""
    if not name:
        return "NA"
    
    import re
    name = name.strip().replace(" ", "_")
    name = re.sub(r'[\/:*?"<>|]', "", name)
    name = re.sub(r"_+", "_", name)
    name = name.strip("_")
    
    return name[:max_len] if name else "NA"

def parse_top_poses(log_path: str, sdf_path: str, n: int = 10) -> list:
    """
    Hybrid Parser: Trích xuất điểm số từ Log và tọa độ từ SDF.
    Giúp tránh crash khi OpenBabel lỗi ghi liên kết nhưng vẫn ghi được tọa độ.
    """
    poses_data = []
    
    # 1. Trích xuất Affinity/CNN từ gnina.log (Dữ liệu đáng tin nhất)
    if os.path.exists(log_path):
        try:
            with open(log_path, 'r') as f:
                start_reading = False
                for line in f:
                    if "-----+------------+------------+------------+----------" in line:
                        start_reading = True
                        continue
                    if start_reading and len(line.strip()) > 10:
                        parts = line.split()
                        if parts[0].isdigit():
                            poses_data.append({
                                "pose_rank": int(parts[0]),
                                "minimizedAffinity": float(parts[1]),
                                "CNNscore": float(parts[3]) if len(parts) > 3 else None,
                                "CNNaffinity": float(parts[4]) if len(parts) > 4 else None
                            })
                    if "done." in line: break
        except Exception as e:
            print(f"⚠️ Error parsing log: {e}")

    # 2. Cố gắng lấy tọa độ/thông tin từ SDF (nếu đọc được)
    try:
        suppl = Chem.SDMolSupplier(sdf_path, removeHs=False, sanitize=False)
        for i, mol in enumerate(suppl):
            if i < len(poses_data) and mol:
                # Có thể thêm các thuộc tính khác từ mol vào poses_data[i] nếu cần
                pass
    except:
        print(f"⚠️ Could not sanitize output SDF, will rely on Log data.")
        
    return poses_data[:n]

def parse_best_score(log_path: str) -> Optional[float]:
    """Lấy affinity tốt nhất trực tiếp từ Log."""
    poses = parse_top_poses(log_path, "", n=1)
    return poses[0]["minimizedAffinity"] if poses else None

print("✅ Utility functions loaded")
# ========================================
# RED FLAG CLASSIFICATION (Adapted from v2.6.2)
# ========================================


# ========================================
# NUCLEOPHILE DATA VALIDATION
# ========================================

def validate_nucleophile_format(nucleophile_data: dict) -> Tuple[bool, str]:
    """
    CRITICAL: Validate nucleophile data format before pipeline execution.
    
    Returns: (is_valid, error_message)
    """
    required_keys = {'chain', 'resnum', 'atom'}
    
    for target, nucs in nucleophile_data.items():
        if not isinstance(nucs, list):
            return (False, f"Target '{target}': Expected list of nucleophiles, got {type(nucs)}")
        
        if len(nucs) == 0:
            print(f"⚠️  Warning: Target '{target}' has no nucleophiles")
            continue
        
        for nuc_idx, nuc in enumerate(nucs):
            if not isinstance(nuc, dict):
                return (False, f"Target '{target}[{nuc_idx}]': Expected dict, got {type(nuc)}")
            
            missing_keys = required_keys - set(nuc.keys())
            if missing_keys:
                return (False, f"Target '{target}[{nuc_idx}]': Missing required keys: {missing_keys}")
            
            # Validate data types
            if not isinstance(nuc['chain'], str):
                return (False, f"Target '{target}[{nuc_idx}]': chain must be string, got {type(nuc['chain'])}")
            if not isinstance(nuc['resnum'], int):
                return (False, f"Target '{target}[{nuc_idx}]': resnum must be int, got {type(nuc['resnum'])}")
            if not isinstance(nuc['atom'], str):
                return (False, f"Target '{target}[{nuc_idx}]': atom must be string, got {type(nuc['atom'])}")
    
    return (True, "")

def classify_red_flag(affinity: Optional[float]) -> str:
    """
    Arbitration Protocol — Tầng 3: Thermodynamic Sanity Check.
    Only checks affinity (not CNN_VS which is a relative ranking metric).
    """
    if affinity is None:
        return "⚪ NO_AFFINITY"
    
    if affinity > 0:
        return "🟡 POSITIVE_AFFINITY"  # Repulsive (unphysical)
    
    if affinity >= AFFINITY_POOR_THRESHOLD:
        return "🟠 POOR_AFFINITY"      # Weak binding
    
    return "✅ GREEN"                  # Good

def classify_covalent_red_flag(
    minimized_affinity: Optional[float],
    bond_distance: Optional[float] = None,
    bond_formation_score: Optional[float] = None
) -> Tuple[str, str]:
    """
    Extended red flag classification for covalent docking.
    Returns: (flag_string, severity_level)
    """
    # Thermodynamic checks (same as flexible)
    if minimized_affinity is None:
        return ("⚪ NO_AFFINITY", "CRITICAL")
    
    if minimized_affinity > 0:
        return ("🟡 POSITIVE_AFFINITY", "CRITICAL")
    
    if minimized_affinity >= AFFINITY_POOR_THRESHOLD:
        return ("🟠 POOR_AFFINITY", "WARNING")
    
    # Covalent-specific checks
    if bond_distance is not None:
        if bond_distance > COVALENT_BOND_DISTANCE_MAX:
            return ("🔴 BOND_TOO_FAR", "WARNING")
        if bond_distance < COVALENT_BOND_DISTANCE_MIN:
            return ("🔴 BOND_TOO_CLOSE", "WARNING")
    
    if bond_formation_score is not None:
        if bond_formation_score < COVALENT_FORMATION_SCORE_MIN:
            return ("🟠 WEAK_COVALENT_GEOMETRY", "WARNING")
    
    return ("✅ GREEN", "PASS")

print("✅ Red flag classification functions loaded")
# ========================================
# COVALENT DOCKING-SPECIFIC FUNCTIONS (NEW)
# ========================================

def identify_target_nucleophiles(
    protein_pdb: str,
    search_residues: List[str] = None,
    auto_detect_radius: float = 10.0
) -> dict:
    """
    Find Cys/Ser/Thr residues suitable for covalent docking.
    
    Args:
        protein_pdb: Path to protein PDB file
        search_residues: List of residue IDs to prioritize (e.g., ["A:123", "A:156"])
        auto_detect_radius: Radius around binding site for auto-detection
    
    Returns:
        Dict: {"Cys123": {"chain": "A", "resnum": 123, "atom": "SG", "xyz": (x,y,z)}, ...}
    """
    nucleophiles = {}
    
    try:
        parser = PDBParser(QUIET=True)
        structure = parser.get_structure("protein", protein_pdb)
        
        # Map 3-letter residue names to single letters
        residue_names = {"CYS": ("Cys", "SG"), "SER": ("Ser", "OG"), "THR": ("Thr", "OG1")}
        
        for model in structure:
            for chain in model:
                for residue in chain:
                    res_name = residue.get_resname().upper()
                    
                    if res_name in residue_names:
                        res_letter, atom_name = residue_names[res_name]
                        res_id = residue.get_id()[1]
                        
                        if atom_name in residue:
                            atom = residue[atom_name]
                            coord = atom.get_coord()
                            
                            key = f"{res_letter}{res_id}"
                            nucleophiles[key] = {
                                "chain": chain.get_id(),
                                "resnum": res_id,
                                "atom": atom_name,
                                "xyz": tuple(coord),
                                "residue_name": res_letter
                            }
    except Exception as e:
        print(f"⚠️ Error parsing nucleophiles from {protein_pdb}: {e}")
    
    return nucleophiles

def check_covalent_bond_formation(
    docked_sdf_path: str,
    protein_pdb: str,
    nucleophile_chain: str,
    nucleophile_resnum: int,
    nucleophile_atom: str,
    distance_min: float = 1.5,
    distance_max: float = 2.2
) -> dict:
    """
    Analyze docked SDF for covalent bond formation by calculating 3D distances.
    CRITICAL: Calculates REAL distances, not hardcoded values!
    
    Returns:
        {"bond_detected": bool, "bond_distance": float, "formation_score": float}
    """
    validation = {
        "bond_detected": False,
        "bond_distance": None,
        "bond_atoms": [],
        "formation_score": 0.0,
    }
    
    try:
        # Step 1: Extract nucleophile 3D coordinates from protein PDB
        nuc_coord = None
        try:
            parser = PDBParser(QUIET=True)
            structure = parser.get_structure("protein", protein_pdb)
            for model in structure:
                chain = model[nucleophile_chain]
                # Standard insertion code tuple: (hetero_flag, resnum, insertion_code)
                res = chain[(' ', nucleophile_resnum, ' ')]
                if nucleophile_atom in res:
                    nuc_coord = res[nucleophile_atom].get_coord()
                    break
        except Exception as pdb_err:
            # Fallback if PDB parsing fails
            return validation
        
        if nuc_coord is None:
            return validation  # Cannot find nucleophile position
        
        # Step 2: Load docked ligand SDF and extract boron coordinates
        suppl = Chem.SDMolSupplier(docked_sdf_path, removeHs=False)
        if not suppl or len(suppl) == 0 or suppl[0] is None:
            return validation
        
        mol = suppl[0]
        if mol.GetNumConformers() == 0:
            return validation
        
        conf = mol.GetConformer(0)  # Top pose
        
        # Step 3: Find boron atoms and calculate distances
        boron_indices = [atom.GetIdx() for atom in mol.GetAtoms() if atom.GetAtomicNum() == 5]
        min_dist = float('inf')
        closest_b_idx = None
        
        for b_idx in boron_indices:
            pos = conf.GetAtomPosition(b_idx)
            b_coord = np.array([pos.x, pos.y, pos.z])
            dist = np.linalg.norm(b_coord - nuc_coord)
            
            if dist < min_dist:
                min_dist = dist
                closest_b_idx = b_idx
        
        # Step 4: Classify bond formation based on distance
        if min_dist < float('inf'):
            validation["bond_distance"] = round(min_dist, 3)
            validation["bond_atoms"] = [f"B{closest_b_idx}"]
            
            # Bond detected if within typical covalent bond range
            validation["bond_detected"] = distance_min <= min_dist <= distance_max
            
            # Calculate formation score (confidence) based on distance quality
            if validation["bond_detected"]:
                # Perfect score (1.0) at ideal distance (midpoint)
                ideal = (distance_min + distance_max) / 2
                deviation = abs(min_dist - ideal) / (distance_max - distance_min)
                validation["formation_score"] = round(max(0, 1.0 - deviation), 3)
            else:
                # Non-zero score even if distance is out of range (shows proximity)
                validation["formation_score"] = 0.0
    
    except Exception as e:
        print(f"⚠️  Error checking covalent bond formation: {e}")
        traceback.print_exc()
    
    return validation

def validate_and_rank_all_poses(
    docked_sdf_path: str,
    protein_pdb: str,
    nucleophile_chain: str,
    nucleophile_resnum: int,
    nucleophile_atom: str,
    distance_min: float = 1.5,
    distance_max: float = 2.4,
    log_path: str = ""
) -> dict:
    
    """
    Best-of-All Pose Selection Logic (v2.7.1 integration).
    
    Process ALL 10 poses from docked SDF:
    1. Calculate distance for EACH pose
    2. Filter to valid poses (1.5-2.4Å gate)
    3. Rank valid poses by CNN_VS score (higher = better for virtual screening)
    4. Return best pose + metadata for all poses
    
    Returns:
        {
            'total_poses': 10,
            'valid_poses': 5,
            'best_pose_rank': 1,  # Rank among ALL 10
            'best_pose_data': {},  # Best pose data dict
            'selected_pose_affinity': -8.5,
            'fallback_used': False
        }
    """
    result = {
        'total_poses': 0,
        'valid_poses': 0,
        'best_pose_rank': None,
        'best_pose_data': {},
        'all_poses_validation': [],
        'selected_pose_affinity': None,
        'fallback_used': False
    }
    
    try:
        # Extract nucleophile coordinates from protein PDB
        nuc_coord = None
        try:
            parser = PDBParser(QUIET=True)
            structure = parser.get_structure("protein", protein_pdb)
            for model in structure:
                chain = model[nucleophile_chain]
                res = chain[(' ', nucleophile_resnum, ' ')]
                if nucleophile_atom in res:
                    nuc_coord = res[nucleophile_atom].get_coord()
                    break
        except Exception:
            return result
        
        if nuc_coord is None:
            return result
        
        # Load ALL poses from docked SDF
        suppl = Chem.SDMolSupplier(docked_sdf_path, removeHs=False)
        if not suppl or len(suppl) == 0:
            return result
        
        result['total_poses'] = len(suppl)
        all_poses = parse_top_poses(log_path, docked_sdf_path, n=10)
        
        # Validate each pose and calculate distances
        valid_poses_with_idx = []
        
        for pose_idx, pose_data in enumerate(all_poses, start=1):
            mol = suppl[pose_idx - 1]
            if mol is None or mol.GetNumConformers() == 0:
                validation = {
                    'pose_rank': pose_idx,
                    'bond_detected': False,
                    'bond_distance': None,
                    'formation_score': 0.0,
                    'valid': False
                }
                result['all_poses_validation'].append(validation)
                continue
            
            # Calculate distance for this pose
            conf = mol.GetConformer(0)
            boron_indices = [atom.GetIdx() for atom in mol.GetAtoms() if atom.GetAtomicNum() == 5]
            
            min_dist = float('inf')
            for b_idx in boron_indices:
                pos = conf.GetAtomPosition(b_idx)
                b_coord = np.array([pos.x, pos.y, pos.z])
                dist = np.linalg.norm(b_coord - nuc_coord)
                if dist < min_dist:
                    min_dist = dist
            
            # Classify this pose
            bond_detected = False
            formation_score = 0.0
            
            if min_dist < float('inf'):
                bond_detected = distance_min <= min_dist <= distance_max
                if bond_detected:
                    ideal = (distance_min + distance_max) / 2
                    deviation = abs(min_dist - ideal) / (distance_max - distance_min)
                    formation_score = round(max(0, 1.0 - deviation), 3)
            
            validation = {
                'pose_rank': pose_idx,
                'bond_detected': bond_detected,
                'bond_distance': round(min_dist, 3) if min_dist < float('inf') else None,
                'formation_score': formation_score,
                'valid': bond_detected
            }
            
            result['all_poses_validation'].append(validation)
            
            if bond_detected:
                cnn_vs = pose_data.get('CNN_VS', float('-inf'))
                valid_poses_with_idx.append((pose_idx, pose_data, validation, cnn_vs))
                result['valid_poses'] += 1
        
        # Select best pose: rank by CNN_VS (higher = better)
        if valid_poses_with_idx:
            valid_poses_with_idx.sort(key=lambda x: x[3], reverse=True)
            best_rank, best_data, _, _ = valid_poses_with_idx[0]
            result['best_pose_rank'] = best_rank
            result['best_pose_data'] = best_data
            result['selected_pose_affinity'] = best_data.get('minimizedAffinity')
        else:
            # Fallback: Use pose #1 if no valid poses
            if len(all_poses) > 0:
                result['best_pose_rank'] = 1
                result['best_pose_data'] = all_poses[0]
                result['selected_pose_affinity'] = all_poses[0].get('minimizedAffinity')
                result['fallback_used'] = True
    
    except Exception as e:
        print(f"⚠️  Error in validate_and_rank_all_poses: {e}")
        traceback.print_exc()
    
    return result

print("✅ Covalent docking-specific functions loaded")
## Cell 3: Command Builders

#Build GNINA commands with covalent-specific flags.
# ========================================
# COVALENT DOCKING COMMAND BUILDER
# ========================================

def build_covalent_docking_command(
    gnina_bin: str,
    receptor_pdb: str,
    ligand_sdf: str,
    ref_ligand_sdf: str,
    flex_residues: List, # Flexible residues (optional, can be None or list)
    covalent_rec_atom: str,           # "A:123:SG" for Cys123
    covalent_lig_pattern: str,        # SMARTS pattern for warhead
    covalent_lig_position: tuple = None,  # (x, y, z) optional
    output_sdf: str = "output.sdf",
    output_flex: str = "flex.pdb",
    log_file: str = "gnina.log",
    num_modes: int = 10,
    exhaustiveness: int = 64,
    gpu_device: str = "0",
    seed: str = "42",
    **kwargs
) -> list:
    """
    Build GNINA command for covalent docking.
    Adds covalent-specific flags to standard docking command.
    
    Based on: d:\\khoa_luan\\covalent_docking_with_GNINA.txt
    """
    cmd = [
        gnina_bin,
        "-r", receptor_pdb,                           # Receptor
        "-l", ligand_sdf,                             # Ligand
        "--autobox_ligand", ref_ligand_sdf,          # Reference for box
        "--autobox_add", str(AUTOBOX_ADD),            # Buffer
        "--autobox_extend", str(AUTOBOX_EXTEND),      # Extend if needed
        "--flexres", flex_residues,                   # Flexible residues
        
        # COVALENT-SPECIFIC FLAGS (NEW)
        "--covalent_rec_atom", covalent_rec_atom,     # Target nucleophile
        "--covalent_lig_atom_pattern", covalent_lig_pattern,  # Warhead SMARTS
        
        # CNN Scoring (same as flexible docking v2.6.2)
        "--cnn_scoring", "none",                  # Post-dock CNN scoring
        "--cnn_empirical_weight", "1.0",             # Empirical weight
        "--pose_sort_order", "Energy",             # Sort by Energy
        "--addH", "0",
        
        # Standard docking parameters
        "--num_modes", str(num_modes),               # Poses per run
        "--exhaustiveness", str(exhaustiveness),     # Search intensity
        "--device", gpu_device,                       # GPU device
        "--seed", str(seed),                          # Reproducibility
        #"--atom_term_data",                           # Embed interaction terms
        
        # Output options
        "-o", output_sdf,                             # Main output
        "--out_flex", output_flex,                    # Flexible residues
        "--full_flex_output",                         # Include full protein in flex
        "--log", log_file,                            # Log file
    ]
    
    # Optional: manual positioning (advanced)
    if covalent_lig_position is not None:
        x, y, z = covalent_lig_position
        cmd.extend([
            "--covalent_lig_atom_position",
            f"{x},{y},{z}"
        ])
    
    # Optional: fix ligand atom position during docking
    if kwargs.get("fix_lig_atom_position", False):
        cmd.append("--covalent_fix_lig_atom_position")
    
    # Optional: bond order (default 1)
    bond_order = kwargs.get("bond_order", 1)
    if bond_order != 1:
        cmd.extend(["--covalent_bond_order", str(bond_order)])
    
    # Optional: UFF optimization of covalent complex
    if kwargs.get("optimize_lig", False):
        cmd.append("--covalent_optimize_lig")
    
    return cmd

print("✅ Covalent command builder loaded")
## Cell 4: Initialization & Data Loading

#Prepare directories, load nucleophile data, and validate inputs.
# ========================================
# INITIALIZE RESULTS DIRECTORIES
# ========================================

def prepare_output_directories():
    """Create base output directory structure."""
    os.makedirs(RESULTS_BASE_DIR, exist_ok=True)
    summary_dir = os.path.join(RESULTS_BASE_DIR, "summary")
    os.makedirs(summary_dir, exist_ok=True)
    print(f"✅ Output directories prepared: {RESULTS_BASE_DIR}")
    return summary_dir

def load_phase2_nucleophile_data() -> dict:
    """Load nucleophile data from PHASE 2 output."""
    try:
        with open(NUCLEOPHILE_DATA_PATH, "r") as f:
            data = json.load(f)
        print(f"✅ Loaded PHASE 2 nucleophile data from {NUCLEOPHILE_DATA_PATH}")
        total_nucleophiles = sum(len(v) for v in data.values())
        print(f"   - {len(data)} targets | {total_nucleophiles} total nucleophiles")
        return data
    except Exception as e:
        print(f"❌ Error loading nucleophile data: {e}")
        return {}

def sanitize_and_load_ligands(input_sdf_path: str) -> list:
    """
    TÍCH HỢP CLEAN: Làm sạch file ligand để tránh lỗi crash GNINA (Kekulize error).
    """
    sanitized_path = input_sdf_path.replace(".sdf", "_super_clean.sdf")
    print(f"🧹 Sanitizing ligands: {os.path.basename(input_sdf_path)}...")
    
    try:
        # 1. Đọc và làm sạch bằng RDKit
        suppl = Chem.SDMolSupplier(input_sdf_path, removeHs=False, sanitize=True)
        writer = Chem.SDWriter(sanitized_path)
        writer.SetKekulize(True) # Ép ghi liên kết đôi rõ ràng thay vì vòng tròn thơm
        
        clean_ligands = []
        count = 0
        
        for i, mol in enumerate(suppl):
            if mol is None: continue
            try:
                # Ép tính lại cấu trúc hóa học chuẩn
                Chem.SanitizeMol(mol)
                Chem.AssignAtomChiralTagsFromStructure(mol)
                
                # Ghi ra file tạm để GNINA sử dụng
                writer.write(mol)
                
                # Lưu vào list để pipeline sử dụng thông tin
                smiles = Chem.MolToSmiles(mol)
                clean_ligands.append({
                    "idx": count + 1,
                    "mol": mol,
                    "smiles": smiles
                })
                count += 1
            except Exception as e:
                print(f"  ⚠️ Conformer {i} failed sanitization: {e}")
        
        writer.close()
        print(f"✅ Created super clean SDF: {os.path.basename(sanitized_path)} ({count} conformers)")
        
        # Cập nhật đường dẫn toàn cục để các bước sau dùng file sạch
        global LIGAND_SDF
        LIGAND_SDF = sanitized_path
        
        return clean_ligands
        
    except Exception as e:
        print(f"❌ Critical error during ligand sanitization: {e}")
        return []

# --- THỰC THI KHỞI TẠO ---
print("\n" + "=" * 80)
print("📋 INITIALIZATION & AUTO-CLEAN")
print("=" * 80)

summary_dir = prepare_output_directories()
nucleophile_data = load_phase2_nucleophile_data()

# Validate nucleophile data
is_valid, err_msg = validate_nucleophile_format(nucleophile_data)
if not is_valid:
    raise ValueError(f"❌ Nucleophile data invalid: {err_msg}")

# Chạy quy trình Super Clean
ligands = sanitize_and_load_ligands(LIGAND_SDF)

print(f"\n✅ Initialization complete. Pipeline ready for {len(ligands)} conformers.")
sanitized_sdf = LIGAND_SDF.replace(".sdf", "_sanitized.sdf")
suppl = Chem.SDMolSupplier(LIGAND_SDF, removeHs=False)
writer = Chem.SDWriter(sanitized_sdf)
for mol in suppl:
    if mol:
        Chem.SanitizeMol(mol) # Ép tính lại bậc liên kết thơm chuẩn RDKit
        writer.write(mol)
writer.close()
LIGAND_SDF = sanitized_sdf # Trỏ pipeline vào file đã sạch
print(f"✅ Sanitized ligand saved to: {LIGAND_SDF}")
print(f"\n✅ Initialization complete")
print(f"   - {len(ligands)} total conformers loaded (S+R stereoisomers)")
print(f"   - {sum(len(v) for v in nucleophile_data.values())} total nucleophiles")
## Cell 5: Main Docking Execution Loop

#Sequential execution of covalent docking runs (target-by-target, conformer-by-conformer).
# ========================================
# GNINA COVALENT DOCKING EXECUTION
# ========================================

def run_gnina_covalent(
    ligand_info: dict,
    target_nucleophile: dict,
    conformer_idx: int,
    output_dir: str,
    receptor_pdb: str,
    run_idx: int,
    total_runs: int
) -> bool:
    """
    Execute single covalent docking run with validation.
    
    Returns:
        True if successful, False otherwise
    """
    lig_id = f"Conf_{conformer_idx:02d}"
    lig_root = os.path.join(output_dir, lig_id)
    os.makedirs(lig_root, exist_ok=True)
    os.makedirs(os.path.join(lig_root, "logs"), exist_ok=True)
    
    # Build paths
    ligand_sdf = os.path.join(lig_root, f"{lig_id}_ligand.sdf")
    output_sdf = os.path.join(lig_root, f"{lig_id}_docked.sdf")
    output_flex = os.path.join(lig_root, f"{lig_id}_flexible_residues.pdb")
    log_file = os.path.join(lig_root, "logs", "gnina.log")
    stderr_file = os.path.join(lig_root, "logs", "gnina_stderr.log")
    command_file = os.path.join(lig_root, "logs", "command.txt")
    
    try:
        # Extract conformer to single-molecule SDF from single ligand file
        suppl = Chem.SDMolSupplier(LIGAND_SDF, removeHs=False, sanitize=False)
        if conformer_idx - 1 < len(suppl) and suppl[conformer_idx - 1] is not None:
            mol = suppl[conformer_idx - 1]
            writer = Chem.SDWriter(ligand_sdf)
            writer.write(mol)
            writer.close()
        else:
            raise RuntimeError(f"Invalid conformer index: {conformer_idx}")
        
        # Build covalent docking command
        covalent_atom = f"{target_nucleophile['chain']}:{target_nucleophile['resnum']}:{target_nucleophile['atom']}"
        
        cmd = build_covalent_docking_command(
            gnina_bin=GNINA_BIN,
            receptor_pdb=receptor_pdb,
            ligand_sdf=ligand_sdf,
            ref_ligand_sdf=REF_LIGAND,
            flex_residues=FLEX_RESIDUES,
            covalent_rec_atom=covalent_atom,
            covalent_lig_pattern=COVALENT_LIG_ATOM_PATTERN,
            covalent_lig_position=COVALENT_LIG_ATOM_POSITION,
            output_sdf=output_sdf,
            output_flex=output_flex,
            log_file=log_file,
            num_modes=NUM_MODES,
            exhaustiveness=EXHAUSTIVENESS,
            gpu_device=GPU_DEVICE,
            seed=SEED,
            fix_lig_atom_position=COVALENT_FIX_LIG_ATOM_POSITION,
            bond_order=COVALENT_BOND_ORDER,
            optimize_lig=COVALENT_OPTIMIZE_LIG,
        )
        # cmd.extend(["--addH", "0"])
        
        # Save command for reproducibility
        with open(command_file, "w") as f:
            f.write(" ".join(cmd) + "\n")
        
        # Write status: RUNNING
        write_status(lig_root, STATUS_RUNNING)
        
        start_time = time.time()
        
        # Execute GNINA
        # Thực thi GNINA
        print(f"🔄 [{run_idx}/{total_runs}] Running {lig_id}...")
        
        try:
            with open(stderr_file, "w") as stderr_f:
                subprocess.run(
                    cmd,
                    check=True,
                    stdout=subprocess.DEVNULL,
                    stderr=stderr_f,
                    text=True,
                    env=_get_subprocess_env(),
                    timeout=GNINA_TIMEOUT_SEC,
                )
            gnina_success = True
        except subprocess.CalledProcessError as e:
            # KIỂM TRA: Nếu có file log và có kết quả bên trong thì vẫn coi là thành công một phần
            if os.path.exists(log_file) and os.path.getsize(log_file) > 0:
                print(f"⚠️ GNINA exited with code {e.returncode} but produced logs. Proceeding...")
                gnina_success = True
            else:
                raise e # Lỗi thực sự nặng
        # Validate output
        if not os.path.exists(output_sdf) or os.path.getsize(output_sdf) == 0:
            raise RuntimeError("GNINA produced empty output SDF")
        
        # BEST-OF-ALL POSE SELECTION (v2.7.1 integration)
        # Validate and rank ALL 10 poses, select best representative
        pose_ranking = validate_and_rank_all_poses(
            docked_sdf_path=output_sdf,
            protein_pdb=receptor_pdb,
            nucleophile_chain=target_nucleophile['chain'],
            nucleophile_resnum=target_nucleophile['resnum'],
            nucleophile_atom=target_nucleophile['atom'],
            distance_min=COVALENT_BOND_DISTANCE_MIN,
            distance_max=COVALENT_BOND_DISTANCE_MAX,
            log_path=log_file
        )
        
        # Extract best affinity from selected pose
        best_affinity = pose_ranking.get('selected_pose_affinity')
        best_pose_rank = pose_ranking.get('best_pose_rank')
        total_poses = pose_ranking.get('total_poses', 0)
        valid_poses = pose_ranking.get('valid_poses', 0)
        
        # For backward compatibility: extract bond info from selected pose
        selected_pose_validation = None
        if best_pose_rank is not None:
            for val in pose_ranking.get('all_poses_validation', []):
                if val.get('pose_rank') == best_pose_rank:
                    selected_pose_validation = val
                    break
        
        elapsed_min = (time.time() - start_time) / 60
        
        # Save comprehensive pose ranking data
        with open(os.path.join(lig_root, "pose_ranking.json"), "w") as f:
            json.dump(pose_ranking, f, indent=2)
        
        # Write status: DONE with multi-pose metadata
        write_status(
            lig_root,
            STATUS_DONE,
            elapsed_min=f"{elapsed_min:.2f}",
            best_affinity=f"{best_affinity:.4f}" if best_affinity is not None else "NA",
            num_poses_total=str(total_poses),
            num_poses_valid=str(valid_poses),
            best_pose_rank=str(best_pose_rank) if best_pose_rank is not None else "NA",
            fallback_used="Y" if pose_ranking.get('fallback_used') else "N",
            covalent_bond_detected="Y" if (selected_pose_validation and selected_pose_validation.get('bond_detected')) else "N",
            bond_distance=f"{selected_pose_validation.get('bond_distance', 'N/A')}" if selected_pose_validation else "N/A",
            formation_score=f"{selected_pose_validation.get('formation_score', 0.0):.2f}" if selected_pose_validation else "0.00",
            target_nucleophile=f"{target_nucleophile['chain']}:{target_nucleophile['resnum']}"
        )
        
        flag, severity = classify_covalent_red_flag(
            best_affinity,
            selected_pose_validation.get('bond_distance') if selected_pose_validation else None,
            selected_pose_validation.get('formation_score') if selected_pose_validation else None
        )
        
        bond_detected_str = selected_pose_validation.get('bond_detected') if selected_pose_validation else False
        print(f"✅ [{run_idx}/{total_runs}] {lig_id} DONE in {elapsed_min:.2f} min | Poses: {total_poses} ({valid_poses} valid, rank {best_pose_rank}) | Affinity: {best_affinity:.2f} | Bond: {bond_detected_str} | Flag: {flag}")
        return True
    
    except subprocess.TimeoutExpired:
        timeout_min = GNINA_TIMEOUT_SEC / 60
        write_status(lig_root, STATUS_FAILED, error=f"TIMEOUT after {timeout_min:.0f}min")
        print(f"⏰ [{run_idx}/{total_runs}] {lig_id} TIMEOUT after {timeout_min:.0f} min")
        return False
    
    except subprocess.CalledProcessError as e:
        write_status(lig_root, STATUS_FAILED, error=f"GNINA exit code {e.returncode}")
        print(f"❌ [{run_idx}/{total_runs}] {lig_id} FAILED (exit code {e.returncode})")
        return False
    
    except Exception as e:
        write_status(
            lig_root,
            STATUS_FAILED,
            error=str(e),
            traceback=traceback.format_exc().replace("\n", " | ")
        )
        print(f"❌ [{run_idx}/{total_runs}] {lig_id} FAILED: {str(e)[:100]}")
        return False

print("✅ Covalent docking execution function loaded")
# ========================================
# MAIN DOCKING LOOP (DEMO - Single Target)
# ========================================
# This cell demonstrates execution for one or more targets.
# Full pipeline would loop over all 9 targets with their nucleophiles

def run_covalent_docking_pipeline(target_id: str = None):
    """
    Execute covalent docking pipeline for specified targets.
    
    Args:
        target_id: Specific target to run, or None for all targets
    """
    print(f"\n" + "=" * 80)
    print(f"🚀 STARTING COVALENT DOCKING PIPELINE")
    print("=" * 80 + "\n")
    
    finished = []
    failed = []
    
    # Get list of targets (9 proteins from PHASE 2)
    targets = list(nucleophile_data.keys())
    if target_id and target_id in targets:
        targets = [target_id]
    
    # Nested loop: target → nucleophile → conformer
    total_runs = sum(
        len(nucleophile_data.get(t, [])) * len(ligands)
        for t in targets
    )
    run_count = 0
    
    for target_name in targets:
        print(f"\n📌 TARGET: {target_name}")
        print("-" * 80)
        
        target_nucleophiles = nucleophile_data.get(target_name, [])
        
        if not target_nucleophiles:
            print(f"⚠️  No nucleophiles found for {target_name}")
            continue
        
        # MANUAL MODE: Filter to only the nucleophile matching COVALENT_REC_ATOM
        # Parse COVALENT_REC_ATOM (format: "CHAIN:RESNUM:ATOM", e.g., "A:248:SG")
        covalent_parts = COVALENT_REC_ATOM.split(':')
        if len(covalent_parts) == 3:
            target_chain, target_resnum, target_atom = covalent_parts[0], int(covalent_parts[1]), covalent_parts[2]
            
            # Filter nucleophiles to match the configured atom
            filtered_nucleophiles = [
                nuc for nuc in target_nucleophiles 
                if nuc.get('chain') == target_chain and 
                   nuc.get('resnum') == target_resnum and 
                   nuc.get('atom') == target_atom
            ]
            
            if filtered_nucleophiles:
                target_nucleophiles = filtered_nucleophiles
                print(f"✓ Filtered to 1 nucleophile matching COVALENT_REC_ATOM={COVALENT_REC_ATOM}")
            else:
                print(f"⚠️  WARNING: No nucleophile found matching COVALENT_REC_ATOM={COVALENT_REC_ATOM}")
                print(f"""   Available nucleophiles: {[f"{n['chain']}:{n['resnum']}:{n['atom']}" for n in target_nucleophiles]}""")
        else:
            print(f"⚠️  WARNING: Could not parse COVALENT_REC_ATOM={COVALENT_REC_ATOM}")
        
        # Get receptor PDB for this target
        protein_path = get_protein_path_for_target(target_name)
        
        # Output directory for this target
        target_output_dir = os.path.join(RESULTS_BASE_DIR, target_name)
        os.makedirs(target_output_dir, exist_ok=True)
        
        # CRITICAL FIX: Iterate through ALL nucleophiles, not just [0]
        for nuc_idx, nucleophile in enumerate(target_nucleophiles):
            nucleophile_dir = os.path.join(target_output_dir, f"nucleophile_{nuc_idx:03d}")
            os.makedirs(nucleophile_dir, exist_ok=True)
            
            # Run docking for each conformer of this nucleophile
            for ligand_info in ligands:
                # ⏩ Skip if already completed
                lig_id = f"Conf_{ligand_info['idx']:02d}"
                lig_root = os.path.join(nucleophile_dir, lig_id)
                if read_status(lig_root) == STATUS_DONE:
                    print(f"⏩ Skipping {lig_id} (Already Done)")
                    run_count += 1
                    continue
                
                run_count += 1
                conformer_idx = ligand_info["idx"]
                
                success = run_gnina_covalent(
                    ligand_info=ligand_info,
                    target_nucleophile=nucleophile,
                    conformer_idx=conformer_idx,
                    output_dir=nucleophile_dir,
                    receptor_pdb=protein_path,
                    run_idx=run_count,
                    total_runs=total_runs
                )
                
                run_id = f"{target_name}_nuc{nuc_idx:03d}_Conf_{conformer_idx:02d}"
                if success:
                    finished.append(run_id)
                else:
                    failed.append(run_id)

    # Final summary
    print(f"\n" + "=" * 80)
    print("📊 DOCKING SUMMARY")
    print("=" * 80)
    print(f"✅ Completed: {len(finished)}")
    print(f"❌ Failed: {len(failed)}")
    print(f"📁 Results: {RESULTS_BASE_DIR}")
    
    if failed and len(failed) <= 10:
        print(f"\n  Failed runs: {', '.join(failed)}")
    elif failed:
        print(f"\n  Failed runs: {', '.join(failed[:10])} ... and {len(failed) - 10} more")
    
    return {"finished": finished, "failed": failed}

# Demo: Run for single target (PPARA)
# Uncomment to execute:
# result = run_covalent_docking_pipeline(target_id="PPARA_7BQ2")
## Cell 6: Progress Monitoring & Tracking

#Track execution progress and generate real-time reports.
# ========================================
# PROGRESS TRACKING
# ========================================

def update_progress_csv(results_dir: str, finished: list, failed: list):
    """Update progress.csv file with current status."""
    summary_dir = os.path.join(results_dir, "summary")
    os.makedirs(summary_dir, exist_ok=True)
    
    progress_file = os.path.join(summary_dir, "progress.csv")
    
    with open(progress_file, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["Status", "Count", "Timestamp"])
        writer.writerow(["COMPLETED", len(finished), datetime.now().isoformat()])
        writer.writerow(["FAILED", len(failed), datetime.now().isoformat()])
        writer.writerow(["TOTAL", len(finished) + len(failed), datetime.now().isoformat()])

def save_completed_list(results_dir: str, finished: list, failed: list):
    """Save lists of completed and failed runs."""
    summary_dir = os.path.join(results_dir, "summary")
    os.makedirs(summary_dir, exist_ok=True)
    
    with open(os.path.join(summary_dir, "finished_runs.txt"), "w") as f:
        for run_id in sorted(finished):
            f.write(f"{run_id}\n")
    
    with open(os.path.join(summary_dir, "failed_runs.txt"), "w") as f:
        for run_id in sorted(failed):
            f.write(f"{run_id}\n")

print("✅ Progress tracking functions loaded")
## Cell 7: Excel Report Generation

#Generate comprehensive Excel summary with covalent-specific metrics.
# ========================================
# EXCEL REPORT GENERATION (Extended for Covalent)
# ========================================

def generate_excel_summary(results_dir: str):
    """
    Generate multi-sheet Excel report of covalent docking results.
    Adapted from flexible docking v2.6.2.
    """
    summary_dir = os.path.join(results_dir, "summary")
    os.makedirs(summary_dir, exist_ok=True)
    
    output_xlsx = os.path.join(summary_dir, "covalent_docking_summary.xlsx")
    
    wb = Workbook()
    
    # Remove default sheet
    if 'Sheet' in wb.sheetnames:
        wb.remove(wb['Sheet'])
    
    # Sheet 1: Poses with Covalent Info (UPDATED with multi-pose metadata)
    ws_poses = wb.create_sheet("Intra-Ligand_Poses", 0)
    ws_poses.append([
        "#", "Conformer", "Target", "Nucleophile",
        "Status", "Num_Poses", "Valid_Poses", "Best_Pose_Rank",
        "Affinity", "CNNscore", "CNN_VS",
        "Covalent_Bond", "Bond_Distance_Å", "Formation_Score",
        "Red_Flag", "SMILES"
    ])
    
    # Sheet 2: Covalent Bond Summary (UPDATED with multi-pose metadata)
    ws_covalent = wb.create_sheet("Covalent_Bonds", 1)
    ws_covalent.append([
        "#", "Conformer", "Target", "Nucleophile",
        "Num_Poses", "Valid_Poses", "Best_Pose_Rank",
        "Bond_Type", "Distance_Å", "Formation_Score",
        "Best_Affinity", "CNNscore", "Geometry_Quality", "SMILES"
    ])
    
    # Sheet 3: Statistics
    ws_stats = wb.create_sheet("Statistics", 2)
    
    # Add statistics
    stats_data = [
        ["=== PIPELINE OVERVIEW ===", ""],
        ["Warhead Type", warhead_type],
        ["Total Docking Runs", "N/A"],  # Count from results
        ["Completed", "N/A"],
        ["Failed", "N/A"],
        ["", ""],
        ["=== MULTI-POSE SELECTION (v2.7.1) ===", ""],
        ["Poses Generated per Run", "10 (GNINA NUM_MODES)"],
        ["Selection Strategy", "Best-of-All (CNN_VS ranking)"],
        ["Fallback Strategy", "Pose #1 (top affinity) if no valid poses"],
        ["Ligands with Valid Covalent Poses", "N/A"],
        ["Average Valid Poses per Run", "N/A"],
        ["Covalent Bond Geometry Gate", f"{COVALENT_BOND_DISTANCE_MIN}-{COVALENT_BOND_DISTANCE_MAX}Å"],
        ["", ""],
        ["=== COVALENT DOCKING ===", ""],
        ["Ligands with Covalent Bonds", "N/A"],
        ["Covalent Bond Success Rate (%)", "N/A"],
        ["Average Bond Distance (Å)", f"{(COVALENT_BOND_DISTANCE_MIN + COVALENT_BOND_DISTANCE_MAX) / 2:.2f}"],
        ["Average Formation Score", "N/A"],
        ["Weak Geometry (score < 0.70)", "N/A"],
        ["", ""],
        ["=== AFFINITY DISTRIBUTION ===", ""],
        ["Excellent (< -8.0 kcal/mol)", "N/A"],
        ["Good (-8.0 to -7.0 kcal/mol)", "N/A"],
        ["Fair (-7.0 to -6.5 kcal/mol)", "N/A"],
        ["Poor (≥ -6.5 kcal/mol)", "N/A"],
    ]
    
    for row_data in stats_data:
        ws_stats.append(row_data)
    
    # Save workbook
    wb.save(output_xlsx)
    print(f"✅ Excel report generated: {output_xlsx}")
    return output_xlsx

print("✅ Excel report generation function loaded")
## Cell 8: Final Summary & Export

#Print final execution summary and export results.
# ========================================
# FINAL SUMMARY & INSTRUCTIONS
# ========================================

print("\n" + "=" * 80)
print("📋 COVALENT DOCKING PIPELINE - USAGE INSTRUCTIONS")
print("=" * 80 + "\n")

usage_instructions = """
## QUICK START

### 1. Pre-Flight Checklist (Run Cell 1 first)
- ✅ GNINA binary found and executable
- ✅ PHASE 1 output loaded: 20 ligand conformers
- ✅ PHASE 2 nucleophile data loaded: 269 nucleophiles
- ✅ All 8 protein PDB files accessible

### 2. Run Covalent Docking for All Targets (Cell 5)
```python
result = run_covalent_docking_pipeline()
```
**Execution Time:** ~3-4 hours (180 runs: 9 targets × 20 conformers × 1 nucleophile each)
**Output:** `results/phase3_docking_results/TARGET_*/`

### 3. Run Docking for Single Target (Cell 5, for testing)
```python
result = run_covalent_docking_pipeline(target_id="PPARA_7BQ2")
```
**Execution Time:** ~15-20 minutes for single target
**Output:** `results/phase3_docking_results/PPARA_7BQ2/`

### 4. Generate Excel Report (Cell 7)
```python
xlsx = generate_excel_summary(RESULTS_BASE_DIR)
```

### 5. Review Results
- **Completed Runs:** `results/phase3_docking_results/summary/finished_runs.txt`
- **Failed Runs:** `results/phase3_docking_results/summary/failed_runs.txt`
- **Excel Report:** `results/phase3_docking_results/summary/covalent_docking_summary.xlsx`

## OUTPUT STRUCTURE

```
results/phase3_docking_results/
├── PPARA_7BQ2/
│   ├── Conf_01/
│   │   ├── STATUS.txt                        # Status and metadata
│   │   ├── Conf_01_ligand.sdf                # Input ligand
│   │   ├── Conf_01_docked.sdf                # GNINA output (all poses)
│   │   ├── Conf_01_flexible_residues.pdb     # Flexible residues
│   │   ├── covalent_validation.json          # Covalent bond info
│   │   └── logs/
│   │       ├── gnina.log                     # GNINA stdout
│   │       ├── gnina_stderr.log              # GNINA stderr
│   │       └── command.txt                   # Exact command executed
│   └── Conf_02/ ... Conf_20/
├── B1_PPARG_8ATY/
│   └── ... (same structure)
├── ... (B1 for 8 targets)
├── B2_PPARA_7BQ2/
│   └── ... (same structure, 8 targets)
├── ... (B2 for remaining targets)
└── summary/
    ├── progress.csv                          # Real-time progress
    ├── finished_runs.txt                     # Completed run IDs
    ├── failed_runs.txt                       # Failed run IDs
    └── covalent_docking_summary_*.xlsx       # Excel reports
```

## KEY PARAMETERS

- **Covalent Bond Distance:** {:.1f}-{:.1f} Å (tunable in .env)
- **Formation Score Threshold:** {:.2f} (confidence > 70%)
- **GNINA Timeout:** {:d}s (~1 hour)
- **GPU Device:** {} (set via CUDA_VISIBLE_DEVICES)
- **Poses per Run:** {} (top 10 extracted)
- **Exhaustiveness:** {} (docking thoroughness)

## TROUBLESHOOTING

**Issue:** GNINA command not found
- **Solution:** Set GNINA_BIN environment variable or ensure gnina is in PATH

**Issue:** Timeout errors (⏰ symbol)
- **Solution:** Increase GNINA_TIMEOUT_SEC in .env (default 3600s)

**Issue:** "Covalent bond not detected"
- **Cause:** Nucleophile poorly positioned or warhead SMARTS incorrect
- **Solution:** Check PHASE 2 nucleophile data, adjust WARHEAD_*_SMARTS patterns

**Issue:** Missing PHASE 1 or PHASE 2 data
- **Solution:** Ensure LIGAND_SDF_WARHEAD_* and NUCLEOPHILE_DATA_PATH are correct in .env

## NEXT PHASE (PHASE 4-6)

Once all 640 runs complete:
1. Aggregate covalent docking results across all targets/warheads
2. Compare B1 vs B2 warhead effectiveness
3. Benchmark against flexible docking results (PHASE 1 baseline)
4. Generate publication-ready HTML report with visualizations
"""

print(usage_instructions.format(
    COVALENT_BOND_DISTANCE_MIN,
    COVALENT_BOND_DISTANCE_MAX,
    COVALENT_FORMATION_SCORE_MIN,
    GNINA_TIMEOUT_SEC,
    GPU_DEVICE,
    NUM_MODES,
    EXHAUSTIVENESS
))

print("\n" + "=" * 80)
print("✅ Pipeline ready for execution!")
print("=" * 80)

In [ ]:
result = run_covalent_docking_pipeline(target_id="PPARA_7BQ2")